In [1]:
!pip install soundfile

In [2]:
!git clone https://github.com/clovaai/aasist.git /kaggle/working/aasist

Cloning into '/kaggle/working/aasist'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 38 (delta 12), reused 6 (delta 6), pack-reused 10 (from 1)
Receiving objects: 100% (38/38), 1.43 MiB | 11.59 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [3]:
# Cell 3: 修复 torchcontrib + 减少 epoch
%cd /kaggle/working/aasist

with open('main.py', 'r') as f:
    content = f.read()

# 修复 SWA import
content = content.replace(
    'from torchcontrib.optim import SWA',
    'from torch.optim.swa_utils import AveragedModel, SWALR, update_bn'
)
content = content.replace(
    'optimizer_swa = SWA(optimizer)',
    'swa_model = AveragedModel(model)\n    swa_scheduler = SWALR(optimizer, swa_lr=optim_config["base_lr"])'
)
content = content.replace(
    'optimizer_swa.update_swa()',
    'swa_model.update_parameters(model)'
)
content = content.replace(
    'optimizer_swa.swap_swa_sgd()',
    '# SWA swap done below'
)
content = content.replace(
    'optimizer_swa.bn_update(trn_loader, model, device=device)',
    'update_bn(trn_loader, swa_model, device=device)\n    model.load_state_dict(swa_model.module.state_dict())'
)

with open('main.py', 'w') as f:
    f.write(content)

# 减少 epoch 到 40
import json
with open('./config/AASIST.conf', 'r') as f:
    config = json.load(f)
config['num_epochs'] = 10
with open('./config/AASIST.conf', 'w') as f:
    json.dump(config, f, indent=4)

print("Done! epochs set to 10")

/kaggle/working/aasist
Done! epochs set to 10


In [4]:
%cd /kaggle/working/aasist

with open('evaluation.py', 'r') as f:
    content = f.read()

content = content.replace('np.float)', 'float)')
content = content.replace('np.int)', 'int)')

with open('evaluation.py', 'w') as f:
    f.write(content)

print("evaluation.py patched!")

/kaggle/working/aasist
evaluation.py patched!


In [5]:
import json

config_path = '/kaggle/working/aasist/config/AASIST.conf'
with open(config_path, 'r') as f:
    config = json.load(f)

config['database_path'] = '/kaggle/input/datasets/anishsarkar22/asvpoof-2019-dataset-la/LA/'
config['num_epochs'] = 10

with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)

print("Config updated!")

Config updated!


In [6]:
%cd /kaggle/working/aasist
!python main.py --config /kaggle/working/aasist/config/AASIST.conf --output_dir /kaggle/working/exp_result

/kaggle/working/aasist
2026-04-02 03:20:43.428363: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775100043.653352      60 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775100043.720202      60 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775100044.269784      60 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775100044.269845      60 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775100044.269849      60 computation_placer.cc:177]